# Dataset Exploration: Quick Overview

This notebook provides a quick exploration of the following datasets:

- **archive_2**: Food.com recipes and interactions
- **archive_3**: Recipe reviews dataset
- **archive_6**: Food coding dataset
- **onlinefoods.csv**: Online food orders dataset

Goal: Understand data structure and prepare for TF-IDF analysis


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set data directory
DATA_DIR = Path('../data')

print("Environment setup complete!")

## 1. Archive_2: Food.com Dataset

Contains recipes and user interactions from Food.com


In [ ]:
# Load archive_2 datasets
archive_2_dir = DATA_DIR / 'archive_2'

print("=" * 80)
print("ARCHIVE_2: Food.com Dataset")
print("=" * 80)

# Load recipes
recipes_a2 = pd.read_csv(archive_2_dir / 'PP_recipes.csv')
print(f"\n📊 Recipes Dataset: {recipes_a2.shape}")
print(f"Columns: {list(recipes_a2.columns)}")
print("\nFirst 3 rows:")
display(recipes_a2.head(3))

# Data types and missing values
print("\n📋 Data Info:")
print(recipes_a2.info())
print("\n🔍 Missing Values:")
print(recipes_a2.isnull().sum())

# Basic statistics
print("\n📈 Numeric Columns Statistics:")
display(recipes_a2.describe())

In [ ]:
# Explore text columns for TF-IDF (if available)
print("\n🔤 Text Column Analysis (for TF-IDF):")

# Check for text columns (common names: name, description, ingredients, steps, tags)
text_cols = [col for col in recipes_a2.columns if any(keyword in col.lower()
             for keyword in ['name', 'description', 'ingredient', 'step', 'tag', 'text'])]

print(f"\nPotential text columns: {text_cols}")

if text_cols:
    for col in text_cols:
        print(f"\n--- {col} ---")
        print(f"Sample values:")
        print(recipes_a2[col].head(3).values)
        if recipes_a2[col].dtype == 'object':
            avg_length = recipes_a2[col].astype(str).str.len().mean()
            print(f"Average length: {avg_length:.1f} characters")

In [ ]:
# Load user interactions
interactions_train = pd.read_csv(archive_2_dir / 'interactions_train.csv')
print(f"\n📊 Interactions Train: {interactions_train.shape}")
display(interactions_train.head(3))

# Rating distribution
if 'rating' in interactions_train.columns:
    print("\n⭐ Rating Distribution:")
    print(interactions_train['rating'].value_counts().sort_index())

    plt.figure(figsize=(10, 4))
    interactions_train['rating'].value_counts(
    ).sort_index().plot(kind='bar', color='skyblue')
    plt.title('Archive_2: Rating Distribution')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.show()

In [ ]:
# Load users data
users_a2 = pd.read_csv(archive_2_dir / 'PP_users.csv')
print(f"\n👥 Users Dataset: {users_a2.shape}")
display(users_a2.head(3))
print(
    f"\nUnique users: {users_a2.shape[0] if 'user_id' in users_a2.columns else 'N/A'}")

## 2. Archive_3: Recipe Reviews Dataset


In [ ]:
# Load archive_3 datasets
archive_3_dir = DATA_DIR / 'archive_3'

print("=" * 80)
print("ARCHIVE_3: Recipe Reviews Dataset")
print("=" * 80)

# Check if parquet or csv is available
if (archive_3_dir / 'recipes.parquet').exists():
    recipes_a3 = pd.read_parquet(archive_3_dir / 'recipes.parquet')
    print("\n✅ Loaded recipes.parquet")
else:
    recipes_a3 = pd.read_csv(archive_3_dir / 'recipes.csv')
    print("\n✅ Loaded recipes.csv")

print(f"\n📊 Recipes Dataset: {recipes_a3.shape}")
print(f"Columns: {list(recipes_a3.columns)}")
display(recipes_a3.head(3))

# Data info
print("\n📋 Data Info:")
print(recipes_a3.info())
print("\n🔍 Missing Values:")
missing = recipes_a3.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Explore text columns for TF-IDF
print("\n🔤 Text Column Analysis (for TF-IDF):")

text_cols_a3 = [col for col in recipes_a3.columns if any(keyword in col.lower()
                for keyword in ['name', 'description', 'ingredient', 'direction', 'tag', 'text', 'title'])]

print(f"Text columns found: {text_cols_a3}")

if text_cols_a3:
    for col in text_cols_a3[:3]:  # Show first 3 text columns
        print(f"\n--- {col} ---")
        print(f"Sample:")
        print(recipes_a3[col].dropna().iloc[0]
              [:200] + "...")  # First 200 chars
        if recipes_a3[col].dtype == 'object':
            avg_length = recipes_a3[col].astype(str).str.len().mean()
            print(f"Average length: {avg_length:.1f} characters")
            print(f"Non-null count: {recipes_a3[col].notna().sum()}")

In [ ]:
# Load reviews
if (archive_3_dir / 'reviews.parquet').exists():
    reviews_a3 = pd.read_parquet(archive_3_dir / 'reviews.parquet')
    print("\n✅ Loaded reviews.parquet")
else:
    reviews_a3 = pd.read_csv(archive_3_dir / 'reviews.csv')
    print("\n✅ Loaded reviews.csv")

print(f"\n📊 Reviews Dataset: {reviews_a3.shape}")
display(reviews_a3.head(3))

# Rating distribution
if 'rating' in reviews_a3.columns or 'Rating' in reviews_a3.columns:
    rating_col = 'rating' if 'rating' in reviews_a3.columns else 'Rating'
    print(f"\n⭐ Rating Distribution:")
    print(reviews_a3[rating_col].value_counts().sort_index())

    plt.figure(figsize=(10, 4))
    reviews_a3[rating_col].value_counts().sort_index().plot(
        kind='bar', color='coral')
    plt.title('Archive_3: Rating Distribution')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.show()

# Check for review text (改进匹配逻辑，排除ID列)
review_text_cols = [col for col in reviews_a3.columns
                    if any(keyword in col.lower() for keyword in ['review', 'text', 'comment', 'feedback'])
                    and 'id' not in col.lower()  # 排除ID列
                    and 'Id' not in col]          # 排除ReviewId等

if review_text_cols:
    print(f"\n💬 Review text columns: {review_text_cols}")

    # 取第一个真正的文本列
    text_col = review_text_cols[0]
    sample_text = reviews_a3[text_col].dropna().iloc[:20]

    print(f"Sample review from '{text_col}':")
    # 检查是否为字符串再切片，避免标量值报错
    if isinstance(sample_text, str):
        print(sample_text[:300])
    else:
        print(f"⚠️ 该列不是文本类型，值为: {sample_text}")
else:
    print("\n⚠️ 未找到评论文本列")
    print(f"可用列: {reviews_a3.columns.tolist()}")

## 3. Archive_6: Food Coding Dataset


In [ ]:
# Load archive_6 dataset
archive_6_dir = DATA_DIR / 'archive_6'

print("=" * 80)
print("ARCHIVE_6: Food Coding Dataset")
print("=" * 80)

food_coded = pd.read_csv(archive_6_dir / 'food_coded.csv')
print(f"\n📊 Food Coded Dataset: {food_coded.shape}")
print(f"Columns: {list(food_coded.columns)}")
display(food_coded.head(3))

# Data info
print("\n📋 Data Info:")
print(food_coded.info())
print("\n🔍 Missing Values:")
missing = food_coded.isnull().sum()
print(missing[missing > 0])

# Statistical summary
print("\n📈 Statistics:")
display(food_coded.describe())

In [ ]:
# Explore categorical columns
print("\n📊 Categorical Columns Distribution:")

categorical_cols = food_coded.select_dtypes(include=['object']).columns
print(f"\nCategorical columns: {list(categorical_cols)}")

for col in categorical_cols[:5]:  # Show first 5 categorical columns
    print(f"\n--- {col} ---")
    print(food_coded[col].value_counts().head(10))

In [ ]:
# Visualize some key features
# Identify potential food preference or coding columns
food_cols = [col for col in food_coded.columns if any(keyword in col.lower()
             for keyword in ['food', 'cuisine', 'comfort', 'fav', 'diet'])]

if food_cols:
    print(f"\n🍽️ Food-related columns: {food_cols[:5]}")

    # Plot first categorical food column
    if len(food_cols) > 0 and food_coded[food_cols[0]].dtype == 'object':
        plt.figure(figsize=(12, 5))
        food_coded[food_cols[0]].value_counts().head(
            15).plot(kind='barh', color='lightgreen')
        plt.title(f'Archive_6: {food_cols[0]} Distribution')
        plt.xlabel('Count')
        plt.tight_layout()
        plt.show()

## 4. Online Foods Dataset


In [ ]:
# Load online foods dataset
print("=" * 80)
print("ONLINE FOODS: Order Dataset")
print("=" * 80)

online_foods = pd.read_csv(DATA_DIR / 'onlinefoods.csv')
print(f"\n📊 Online Foods Dataset: {online_foods.shape}")
print(f"Columns: {list(online_foods.columns)}")
display(online_foods.head(3))

# Data info
print("\n📋 Data Info:")
print(online_foods.info())
print("\n🔍 Missing Values:")
print(online_foods.isnull().sum())

# Statistics
print("\n📈 Statistics:")
display(online_foods.describe())

In [ ]:
# Explore categorical features
print("\n📊 Categorical Features:")

categorical_cols_online = online_foods.select_dtypes(
    include=['object']).columns
print(f"Categorical columns: {list(categorical_cols_online)}")

for col in categorical_cols_online:
    print(f"\n--- {col} ---")
    print(f"Unique values: {online_foods[col].nunique()}")
    print(online_foods[col].value_counts().head(10))

In [ ]:
# Visualizations for online foods
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Distribution of a key categorical variable
if len(categorical_cols_online) > 0:
    col1 = categorical_cols_online[0]
    online_foods[col1].value_counts().head(10).plot(
        kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title(f'{col1} Distribution')
    axes[0].set_xlabel(col1)
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Another categorical variable
if len(categorical_cols_online) > 1:
    col2 = categorical_cols_online[1]
    online_foods[col2].value_counts().head(10).plot(
        kind='bar', ax=axes[1], color='salmon')
    axes[1].set_title(f'{col2} Distribution')
    axes[1].set_xlabel(col2)
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Summary and TF-IDF Preparation

### Dataset Overview Summary


In [ ]:
# Create summary table
summary_data = {
    'Dataset': ['Archive_2 Recipes', 'Archive_2 Interactions', 'Archive_3 Recipes',
                'Archive_3 Reviews', 'Archive_6 Food Coded', 'Online Foods'],
    'Rows': [recipes_a2.shape[0], interactions_train.shape[0], recipes_a3.shape[0],
             reviews_a3.shape[0], food_coded.shape[0], online_foods.shape[0]],
    'Columns': [recipes_a2.shape[1], interactions_train.shape[1], recipes_a3.shape[1],
                reviews_a3.shape[1], food_coded.shape[1], online_foods.shape[1]]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 80)
print("📊 DATASET SUMMARY")
print("=" * 80)
display(summary_df)

# Visualize summary
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(summary_df['Dataset']))
width = 0.35

bars = ax.bar(x, summary_df['Rows'], width, label='Rows', color='skyblue')
ax.set_xlabel('Dataset')
ax.set_ylabel('Count')
ax.set_title('Dataset Size Overview')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Dataset'], rotation=45, ha='right')
ax.legend()
ax.set_yscale('log')  # Log scale for better visualization

plt.tight_layout()
plt.show()

### Text Columns Identified for TF-IDF Analysis


In [ ]:
print("\n" + "=" * 80)
print("🔤 TEXT COLUMNS FOR TF-IDF ANALYSIS")
print("=" * 80)

tfidf_candidates = {
    'Archive_2 Recipes': [col for col in recipes_a2.columns if recipes_a2[col].dtype == 'object'],
    'Archive_3 Recipes': [col for col in recipes_a3.columns if recipes_a3[col].dtype == 'object'],
    'Archive_3 Reviews': [col for col in reviews_a3.columns if reviews_a3[col].dtype == 'object'],
    'Archive_6 Food': [col for col in food_coded.columns if food_coded[col].dtype == 'object'],
    'Online Foods': [col for col in online_foods.columns if online_foods[col].dtype == 'object']
}

for dataset_name, columns in tfidf_candidates.items():
    print(f"\n{dataset_name}:")
    for col in columns[:5]:  # Show first 5
        print(f"  - {col}")
    if len(columns) > 5:
        print(f"  ... and {len(columns) - 5} more columns")

### Next Steps for TF-IDF Analysis

Based on the exploration, here are the recommended text columns for TF-IDF:

1. **Archive_2**: Recipe names, ingredients, tags, or steps
2. **Archive_3**: Recipe names/titles, ingredients, directions, review text
3. **Archive_6**: Food preference columns, dietary information
4. **Online Foods**: Food names, feedback text (if available)

**Next notebook**: `02_tfidf_analysis.ipynb` will focus on:

- Text preprocessing (cleaning, tokenization)
- TF-IDF vectorization
- Feature extraction and analysis
- Similarity calculations


In [ ]:
# Save key dataframes for next notebook
print("\n💾 Datasets loaded and ready for TF-IDF analysis!")
print("\nDataframes available in this session:")
print("  - recipes_a2 (Archive_2 recipes)")
print("  - interactions_train (Archive_2 interactions)")
print("  - recipes_a3 (Archive_3 recipes)")
print("  - reviews_a3 (Archive_3 reviews)")
print("  - food_coded (Archive_6)")
print("  - online_foods (Online foods orders)")